# White Hat / Black Hat: Country Recovery Fairness

This notebook focuses on the country-level ethics pair used in the final webpage. Both visualizations use the same state-level emissions dataset and the same 2019-to-2024 comparison. The white-hat version asks whether each country's CO₂ recovery was proportional to its traffic recovery. The black-hat version demonstrates how the same data can be narrowed to percent flight growth only, creating a cleaner but incomplete recovery narrative.


## 1. Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

In [ ]:
def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "data").exists():
            return path
    raise RuntimeError("Could not find project root")


PROJECT_ROOT = find_project_root()
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)


def save_figure(fig, name: str) -> None:
    fig.savefig(FIGURE_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{name}.pdf", bbox_inches="tight")

In [ ]:
# palette from european_aviation_final.html design tokens
BLUE  = "#537D96"
RED   = "#BF4646"
DARK  = "#2C2F33"
MUTED = "#72777C"
LINE  = "#E3E6E8"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.edgecolor":   LINE,
    "axes.linewidth":   0.8,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.labelcolor":  DARK,
    "axes.labelsize":   9,
    "axes.labelpad":    8,
    "axes.titlesize":   14,
    "axes.titleweight": "bold",
    "axes.titlecolor":  DARK,
    "axes.titlepad":    12,
    "xtick.color":      MUTED,
    "xtick.labelsize":  9,
    "ytick.color":      MUTED,
    "ytick.labelsize":  9,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "font.family":      "sans-serif",
    "figure.dpi":       130,
})


## 2. Data


In [ ]:
df_state = pd.read_csv(CLEAN_DIR / "emission_state_clean.csv", parse_dates=["DATE"])

CODE2NAME = {
    "AL": "Albania", "AM": "Armenia", "AT": "Austria", "AZ": "Azerbaijan",
    "BA": "Bosnia", "BE": "Belgium", "BG": "Bulgaria", "CH": "Switzerland",
    "CY": "Cyprus", "CZ": "Czechia", "DE": "Germany", "DK": "Denmark",
    "EE": "Estonia", "ES": "Spain", "FI": "Finland", "FR": "France",
    "GB": "UK", "GE": "Georgia", "GR": "Greece", "HR": "Croatia",
    "HU": "Hungary", "IE": "Ireland", "IL": "Israel", "IS": "Iceland",
    "IT": "Italy", "LT": "Lithuania", "LU": "Luxembourg", "LV": "Latvia",
    "MA": "Morocco", "MD": "Moldova", "ME": "Montenegro", "MK": "Macedonia",
    "MT": "Malta", "NL": "Netherlands", "NO": "Norway", "PL": "Poland",
    "PT - Lisbon FIR": "Portugal", "PT - Santa Maria FIR": "PT Azores",
    "RO": "Romania", "RS": "Serbia", "SE": "Sweden", "SI": "Slovenia",
    "SK": "Slovakia", "TR": "Turkey", "UA": "Ukraine",
}


## 3. Pair C — Country Recovery Fairness

The central project question is whether European aviation recovered sustainably after COVID-19. At country level, this is not just a question of which markets grew fastest. A fair comparison must ask two things at the same time: how much traffic came back, and whether CO₂ came back faster than traffic.

The white-hat visualization uses a two-dimensional benchmark. Each country is positioned by its 2024 flight recovery on the x-axis and its 2024 CO₂ recovery on the y-axis, both indexed to 2019. The diagonal reference line marks proportional recovery: countries above it have an emissions penalty because CO₂ rose more than traffic; countries below it recovered traffic with a lower CO₂ rebound. Bubble size adds the missing scale context by encoding absolute 2024 CO₂.

The black-hat visualization intentionally removes that context. It ranks countries only by percent flight growth above 2019. This can make small or unusual markets look like the main recovery story while hiding whether emissions grew faster than traffic and how much CO₂ each country actually contributed.


In [ ]:
# ── data ──────────────────────────────────────────────────────────────
state_year = (
    df_state.groupby(["AREA", "YEAR"])
    .agg(flights=("NB_FLIGHTS", "sum"), co2=("CO2_KG", "sum"))
    .reset_index()
)
wide = (
    state_year[state_year["YEAR"].isin([2019, 2024])]
    .pivot(index="AREA", columns="YEAR", values=["flights", "co2"])
    .dropna()
)
state_piv = pd.DataFrame({
    "AREA":        wide.index.astype(str),
    "flight_idx":  wide["flights"][2024] / wide["flights"][2019] * 100,
    "co2_idx":     wide["co2"][2024]     / wide["co2"][2019]     * 100,
    "co2_2024_mt": wide["co2"][2024] / 1e9,
})
state_piv["gap"]  = state_piv["co2_idx"] - state_piv["flight_idx"]
state_piv["name"] = state_piv["AREA"].map(CODE2NAME).fillna(state_piv["AREA"])

# keep only countries inside the visible domain (excludes Ukraine outlier)
ps = state_piv[(state_piv["flight_idx"].between(45, 185)) &
               (state_piv["co2_idx"].between(45, 185))].copy()

# ── plot ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 8))

LIM = (45, 185)
ax.set_xlim(*LIM)
ax.set_ylim(*LIM)

# quadrant fills
ax.fill_between([LIM[0], LIM[1]], [LIM[0], LIM[1]], LIM[1],
                color=RED, alpha=0.04, zorder=0)
ax.fill_between([LIM[0], LIM[1]], LIM[0], [LIM[0], LIM[1]],
                color=BLUE, alpha=0.04, zorder=0)

# 2019 baseline rules
ax.axvline(100, color=MUTED, lw=0.8, alpha=0.5, zorder=1)
ax.text(101, 183, "2019 baseline", color=MUTED, fontsize=9, va="top")

# diagonal parity line
ax.plot(LIM, LIM, color=DARK, ls="--", lw=1.3, zorder=2)
ax.text(125, 129, "CO₂ = traffic recovery",
        color=DARK, fontsize=9, fontstyle="italic", rotation=40, va="bottom")

# zone callouts
ax.text(47, 183, "Emissions outpace traffic",
        color=RED, fontsize=10, fontweight="bold", va="top")
ax.text(183, 47, "Traffic outpaces emissions",
        color=BLUE, fontsize=10, fontweight="bold", ha="right", va="bottom")

# bubbles
colors = np.where(ps["gap"] >= 0, RED, BLUE)
sizes  = (ps["co2_2024_mt"].clip(lower=0.05) ** 0.55) * 160
sc = ax.scatter(ps["flight_idx"], ps["co2_idx"],
                s=sizes, c=colors, alpha=0.78,
                edgecolors="white", linewidths=0.9, zorder=3)

# country labels — bold for top emitters, regular for the rest
top8 = set(ps.nlargest(8, "co2_2024_mt")["name"])
for _, r in ps.iterrows():
    bold = r["name"] in top8
    ax.annotate(r["name"], (r["flight_idx"], r["co2_idx"]),
                xytext=(0, -6), textcoords="offset points",
                ha="center", va="top",
                fontsize=8 if bold else 7,
                color=DARK if bold else MUTED,
                fontweight="bold" if bold else "normal")

ax.set_xlabel("2024 flights vs 2019 (%)")
ax.set_ylabel("2024 CO₂ vs 2019 (%)")
ax.set_title(
    "Recovery is not even: some countries pay an emissions penalty",
)

fig.tight_layout()
save_figure(fig, "whitehat_country_recovery_gap_scatter")
plt.show()

**White-hat explanation.** This plot preserves the analytical question: did emissions recover proportionally with traffic? The shared scale on both axes makes the diagonal meaningful. A country above the diagonal is environmentally worse than its traffic recovery alone suggests; a country below the diagonal has a relatively cleaner recovery. Bubble size prevents percentage outliers from dominating the interpretation, because absolute 2024 CO₂ remains visible. This is the more ethical framing because it combines relative change, emissions efficiency, and climate burden in one chart.

**How to read it.** Start at the 100% vertical and horizontal reference lines, which mark the 2019 baseline. Then compare each point to the diagonal. The farther a country sits above the diagonal, the more its CO₂ rebound outpaced its flight rebound. Large bubbles above the line deserve particular attention because they combine a worsening emissions profile with substantial absolute emissions.


In [ ]:
# ── data ──────────────────────────────────────────────────────────────
df = (
    state_piv.sort_values("flight_idx", ascending=False)
    .head(18)
    .reset_index(drop=True)
)
df["growth"] = df["flight_idx"] - 100
N_RED = 3
colors = [RED if i < N_RED else BLUE for i in range(len(df))]

# ── plot ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
fig.subplots_adjust(left=0.40, top=0.86, right=0.95, bottom=0.1)

bars = ax.barh(df["name"], df["growth"], color=colors, height=0.72)

# value labels
for bar, val in zip(bars, df["growth"]):
    ax.text(bar.get_width() + df["growth"].max() * 0.012,
            bar.get_y() + bar.get_height() / 2,
            f"+{val:.0f}%", va="center", fontsize=9,
            color=DARK, fontweight="bold")

ax.axvline(0, color=DARK, lw=1.2)

# dashed separator between the two groups
sep_y = (bars[N_RED - 1].get_y() + bars[N_RED].get_y() + bars[N_RED].get_height()) / 2

# ── group labels + bracket lines in the left margin ──────────────────
trans = ax.get_yaxis_transform()

def _group_label(y_top, y_bot, text, color):
    ax.plot([-0.17, -0.17], [y_top, y_bot],
            transform=trans, color=color, lw=4,
            clip_on=False, solid_capstyle="butt")
    ax.text(-0.19, (y_top + y_bot) / 2, text,
            transform=trans, ha="right", va="center",
            fontsize=9, color=color, fontweight="bold",
            linespacing=1.4, clip_on=False)

_group_label(bars[0].get_y(),
             bars[N_RED - 1].get_y() + bars[N_RED - 1].get_height(),
             "Top 3    \nfastest   \ngrowers  ", RED)
_group_label(bars[N_RED].get_y(),
             bars[-1].get_y() + bars[-1].get_height(),
             "Other    \nrecoveries", BLUE)

ax.invert_yaxis()
ax.set_xlabel("Growth above 2019 (%)")
ax.set_xlim(left=0)
fig.suptitle("The aviation recovery is led by fast-growing markets",
             x=0.6, fontsize=14, fontweight="bold", color=DARK, y=0.96)
fig.text(0.6, 0.90,
         "Selective framing: ignores absolute CO\u2082 size and whether emissions outpaced flights.",
         ha="center", fontsize=9, color=MUTED)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)

save_figure(fig, "blackhat_country_recovery_percent_only")
plt.show()


**Black-hat explanation.** This plot is misleading because it changes the question from environmental recovery to traffic expansion. The bars are based on real values, but the chart hides CO₂ recovery, CO₂ per flight, and absolute emissions mass. As a result, countries with very high percentage growth can appear to lead the aviation recovery even if their total climate impact is much smaller than that of major emitting markets.

**Manipulation used.** The chart ranks only countries with the largest flight-growth percentages and highlights the top three in red. That visual emphasis invites the reader to treat growth as importance. The subtitle admits the omitted context, but the encoding itself still pushes a selective story: fast-growing markets look central, while larger emitters and countries where CO₂ outpaced traffic are pushed out of view.
